# 02 — Train the sign-to-speech model

Run after `01_filter_vocabulary.ipynb`. Settings: GPU T4, `asl-signs` attached.

Design follows what notebook 01 measured:

| Measurement | Consequence |
|---|---|
| clip length p10 6, median 25, p90 96, max 269 | **resample** every clip to a fixed 32 frames instead of pad/crop — a 30-frame crop would destroy both ends of the distribution |
| 30% of frames have no hand detected | explicit per-hand **presence flag**, missing coords zeroed |
| 21 participants only | 3 held-out signers, so report accuracy as an estimate with few test signers |
| 346–402 clips per word | classes are balanced; plain accuracy is meaningful |

Outputs to `/kaggle/working/`: `signbridge.tflite`, `preprocess_config.json`, `confusion_matrix.png`, `metrics.json`.

In [ ]:
import glob, json, os, time
import numpy as np, pandas as pd
import tensorflow as tf

hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
assert hits, 'Attach the asl-signs data: Add Input -> Competitions -> asl-signs'
DATA = os.path.dirname(hits[0])
OUT  = '/kaggle/working'
print('DATA =', DATA, '| GPU:', tf.config.list_physical_devices('GPU'))

## Configuration

Every constant here must be mirrored exactly in the Kotlin preprocessing. Mismatched normalization is silent: the model trains fine and fails on the phone.

In [ ]:
WORDS = ['hello','thankyou','please','bye','yes','no','mom','dad',
         'hungry','drink','potty','owie','sick','hot','sleep',
         'wait','go','finish','mad','sad']

T          = 32    # frames per clip after resampling
N_HAND     = 21    # MediaPipe hand landmarks
POSE_LSH   = 11    # pose landmark: left shoulder
POSE_RSH   = 12    # pose landmark: right shoulder
FEAT       = 2 * (N_HAND * 2 + 2 + 1)   # per hand: 42 coords + 2 body-relative + 1 presence
TEST_IDS   = [30680, 37779, 49445]      # held-out signers, from notebook 01
SEED       = 42

np.random.seed(SEED); tf.random.set_seed(SEED)
print('T =', T, ' FEAT =', FEAT)

## Preprocessing

Per frame and per hand:

1. translate so the wrist (landmark 0) sits at the origin
2. scale by the largest wrist-to-landmark distance, so hand size and camera distance drop out
3. keep the wrist position **relative to the shoulder midpoint**, scaled by shoulder width — this is what tells the model *where* the sign happens (`mom` at the chin vs `dad` at the forehead). Wrist-only normalization erases it.
4. a presence flag, 0 when that hand is missing in that frame

Then resample the clip's frames to exactly `T` by linear index sampling, which both repeats frames for a 6-frame clip and subsamples a 269-frame one.

In [ ]:
USE = ['left_hand','right_hand','pose']

def clip_features(path):
    df = pd.read_parquet(f'{DATA}/{path}', columns=['frame','type','landmark_index','x','y'])
    df = df[df['type'].isin(USE)]
    frames = np.sort(df['frame'].unique())
    if len(frames) == 0:
        return np.zeros((T, FEAT), np.float32)

    # frame -> row block, as dense arrays
    g = {k: v for k, v in df.groupby('type')}
    def grid(t, n):
        sub = g.get(t)
        a = np.full((len(frames), n, 2), np.nan, np.float32)
        if sub is None:
            return a
        fi = np.searchsorted(frames, sub['frame'].to_numpy())
        li = sub['landmark_index'].to_numpy()
        ok = li < n
        a[fi[ok], li[ok], 0] = sub['x'].to_numpy()[ok]
        a[fi[ok], li[ok], 1] = sub['y'].to_numpy()[ok]
        return a

    lh, rh = grid('left_hand', N_HAND), grid('right_hand', N_HAND)
    pose   = grid('pose', 33)

    # body reference: shoulder midpoint and shoulder width
    lsh, rsh = pose[:, POSE_LSH], pose[:, POSE_RSH]
    mid   = (lsh + rsh) / 2.0
    width = np.linalg.norm(lsh - rsh, axis=-1)
    bad   = ~np.isfinite(width) | (width < 1e-6)
    width = np.where(bad, 1.0, width)
    mid   = np.where(np.isfinite(mid), mid, 0.0)

    feats = []
    for hand in (lh, rh):
        wrist   = hand[:, 0:1, :]                       # (F,1,2)
        present = np.isfinite(wrist).all(-1).squeeze(-1)  # (F,)
        rel     = hand - wrist
        scale   = np.nanmax(np.linalg.norm(rel, axis=-1), axis=-1)   # (F,)
        scale   = np.where(np.isfinite(scale) & (scale > 1e-6), scale, 1.0)
        rel     = rel / scale[:, None, None]
        rel     = np.where(np.isfinite(rel), rel, 0.0)

        loc = (wrist.squeeze(1) - mid) / width[:, None]  # wrist relative to body
        loc = np.where(np.isfinite(loc), loc, 0.0)

        feats.append(np.concatenate([
            rel.reshape(len(frames), -1),               # 42
            loc,                                        # 2
            present[:, None].astype(np.float32),        # 1
        ], axis=1))

    x = np.concatenate(feats, axis=1).astype(np.float32)   # (F, FEAT)

    idx = np.linspace(0, len(frames) - 1, T).round().astype(int)
    return x[idx]

In [ ]:
# Warning: ~7600 parquet reads. Expect 5-10 minutes. Cached to .npy so a rerun is instant.
sub = pd.read_csv(f'{OUT}/subset.csv') if os.path.exists(f'{OUT}/subset.csv') else None
if sub is None:
    tr = pd.read_csv(f'{DATA}/train.csv')
    sub = tr[tr['sign'].isin(WORDS)].copy()
print(len(sub), 'clips')

CACHE = f'{OUT}/features.npz'
if os.path.exists(CACHE):
    z = np.load(CACHE); X, y, pid = z['X'], z['y'], z['pid']
    print('loaded cache', X.shape)
else:
    t0 = time.time(); X = np.zeros((len(sub), T, FEAT), np.float32)
    for i, p in enumerate(sub['path'].to_numpy()):
        X[i] = clip_features(p)
        if i % 500 == 0:
            print(f'{i}/{len(sub)}  {time.time()-t0:.0f}s', flush=True)
    w2i = {w: i for i, w in enumerate(WORDS)}
    y   = sub['sign'].map(w2i).to_numpy().astype(np.int32)
    pid = sub['participant_id'].to_numpy()
    np.savez_compressed(CACHE, X=X, y=y, pid=pid)
    print('done', X.shape, f'{time.time()-t0:.0f}s')

In [ ]:
is_test = np.isin(pid, TEST_IDS)
Xtr, ytr = X[~is_test], y[~is_test]
Xte, yte = X[is_test],  y[is_test]
print('train', Xtr.shape, ' test', Xte.shape)
print('test signers:', sorted(set(pid[is_test].tolist())))

# carve a validation set out of TRAIN signers, so the test signers stay untouched
tr_people = np.array(sorted(set(pid[~is_test].tolist())))
rng = np.random.default_rng(SEED); rng.shuffle(tr_people)
val_ids = set(tr_people[:3].tolist())
is_val  = np.isin(pid[~is_test], list(val_ids))
Xva, yva = Xtr[is_val], ytr[is_val]
Xtr2, ytr2 = Xtr[~is_val], ytr[~is_val]
print('fit', Xtr2.shape, ' val', Xva.shape, ' val signers:', sorted(val_ids))

## Model

A small bidirectional GRU. Kept narrow on purpose: it has to run on a phone, and the whole point of the measurement is what fits on-device.

In [ ]:
def build():
    inp = tf.keras.Input((T, FEAT))
    # unroll=True: the clip length is fixed at T, and unrolling keeps the export
    # to plain TFLite builtin ops. A rolled GRU emits TensorList ops that need the
    # Flex delegate, which drags a large extra dependency into the APK.
    x = tf.keras.layers.Bidirectional(
            tf.keras.layers.GRU(96, return_sequences=True, unroll=True))(inp)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(96, unroll=True))(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    out = tf.keras.layers.Dense(len(WORDS), activation='softmax')(x)
    m = tf.keras.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

model = build(); model.summary()

In [ ]:
cb = [
    tf.keras.callbacks.EarlyStopping('val_accuracy', patience=12, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=5, min_lr=1e-5),
]
hist = model.fit(Xtr2, ytr2, validation_data=(Xva, yva),
                 epochs=80, batch_size=64, callbacks=cb, verbose=2)

## Evaluation on held-out signers

This number means *works on a person the model never saw*, which is the number that matters for the demo.

In [ ]:
loss, acc = model.evaluate(Xte, yte, verbose=0)
prob = model.predict(Xte, verbose=0)
pred = prob.argmax(1)
top3 = np.mean([yte[i] in np.argsort(prob[i])[-3:] for i in range(len(yte))])
print(f'held-out signer accuracy : {acc:.3f}')
print(f'top-3 accuracy           : {top3:.3f}')
print(f'chance                   : {1/len(WORDS):.3f}')

In [ ]:
import matplotlib.pyplot as plt
cm = tf.math.confusion_matrix(yte, pred, num_classes=len(WORDS)).numpy()
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(WORDS)), WORDS, rotation=90)
ax.set_yticks(range(len(WORDS)), WORDS)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title(f'SignBridge — held-out signers (n={len(yte)}), accuracy {acc:.1%}')
for i in range(len(WORDS)):
    for j in range(len(WORDS)):
        if cmn[i, j] > 0.02:
            ax.text(j, i, f'{cmn[i,j]*100:.0f}', ha='center', va='center',
                    fontsize=7, color='white' if cmn[i, j] > .5 else 'black')
fig.colorbar(im, shrink=.8); fig.tight_layout()
fig.savefig(f'{OUT}/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
per = pd.DataFrame({
    'word': WORDS,
    'n': cm.sum(1),
    'recall': np.round(np.diag(cmn), 3),
}).sort_values('recall')
print('weakest words:'); print(per.head(6).to_string(index=False))

print('\nmost confused pairs:')
pairs = [(WORDS[i], WORDS[j], cmn[i, j]) for i in range(len(WORDS))
         for j in range(len(WORDS)) if i != j and cmn[i, j] > 0.08]
for a, b, v in sorted(pairs, key=lambda t: -t[2])[:8]:
    print(f'  {a:10s} mistaken for {b:10s} {v:.0%}')

## Export for Android

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]  # builtins only, no Flex
tfl = conv.convert()
open(f'{OUT}/signbridge.tflite', 'wb').write(tfl)
print('tflite size: %.2f MB' % (len(tfl) / 1e6))

# verify the exported model matches Keras on real inputs
it = tf.lite.Interpreter(model_content=tfl); it.allocate_tensors()
i0, o0 = it.get_input_details()[0], it.get_output_details()[0]
agree = 0
for k in range(min(50, len(Xte))):
    it.set_tensor(i0['index'], Xte[k:k+1]); it.invoke()
    agree += int(it.get_tensor(o0['index'])[0].argmax() == pred[k])
print(f'tflite agrees with keras on {agree}/50 test clips')

In [ ]:
cfg = {
  'words': WORDS, 'frames': T, 'features': FEAT,
  'hand_landmarks': N_HAND, 'pose_left_shoulder': POSE_LSH, 'pose_right_shoulder': POSE_RSH,
  'feature_order': 'per frame: [left_hand 42 normalized xy, left wrist 2 body-relative, '
                   'left presence 1, right_hand 42, right wrist 2, right presence 1]',
  'normalization': 'hand: subtract wrist(landmark 0), divide by max |landmark-wrist| in that frame. '
                   'body-relative: (wrist - shoulder_midpoint) / shoulder_width. '
                   'missing values -> 0.0, presence flag -> 0.0',
  'resampling': 'np.linspace(0, n_frames-1, T).round() index sampling',
  'test_signers': TEST_IDS,
}
json.dump(cfg, open(f'{OUT}/preprocess_config.json','w'), indent=2)

metrics = {'heldout_signer_accuracy': float(acc), 'top3_accuracy': float(top3),
           'n_test_clips': int(len(yte)), 'n_test_signers': len(TEST_IDS),
           'n_train_clips': int(len(Xtr2)), 'tflite_mb': round(len(tfl)/1e6, 2),
           'per_word_recall': {w: float(r) for w, r in zip(WORDS, np.diag(cmn))}}
json.dump(metrics, open(f'{OUT}/metrics.json','w'), indent=2)
print(json.dumps(metrics, indent=2)[:600])

Download from the **Output** panel on the right: `signbridge.tflite`, `preprocess_config.json`, `confusion_matrix.png`, `metrics.json`.

`preprocess_config.json` is the contract for the Kotlin side. The Android preprocessing has to reproduce `normalization`, `feature_order` and `resampling` exactly.